# Phase 3 Testing — Model Inference Validation

This notebook tests the **full prediction pipeline**: raw transaction → feature engineering → LightGBM model → fraud verdict.

**Why test inference?**
Even if artefacts load correctly (Phase 1) and feature engineering is mathematically sound (Phase 2), the model itself could still behave incorrectly:
- It might return probabilities outside `[0, 1]`
- It might give different answers to the same input (non-determinism)
- It might score obviously fraudulent transactions *lower* than legitimate ones (directional failure)
- It might crash on a batch of inputs instead of a single row

These tests are the final quality gate before the model is considered safe to deploy.

**Scope:** No Flask, no API calls. The full chain is: `engineer_features()` → `model.predict_proba()` → assertions.

**How to read results:** Each test prints `✅ PASS` or `❌ FAIL`. All 5 must pass.

---
| Phase | Notebook | Tests |
|-------|----------|-------|
| Phase 1 | `Phase_1_testing.ipynb` | Artefact loading (Tests 1–5) |
| Phase 2 | `Phase_2_testing.ipynb` | Feature engineering math (Tests 6–14) |
| **Phase 3** | **this notebook** | **Model inference quality (Tests 15–19)** |

In [1]:
# ── Setup: load artefacts and define pipeline ─────────────────────────────────
import os
import math
import numpy as np
import pandas as pd
import joblib

ARTEFACT_DIR = os.path.join(os.getcwd(), "model_artefacts")
model        = joblib.load(os.path.join(ARTEFACT_DIR, "lgbm_fraud_model.pkl"))
scaler       = joblib.load(os.path.join(ARTEFACT_DIR, "scaler.pkl"))
feature_cols = joblib.load(os.path.join(ARTEFACT_DIR, "feature_columns.pkl"))
num_features = joblib.load(os.path.join(ARTEFACT_DIR, "num_features.pkl"))

def check(condition, pass_msg, fail_msg):
    """Simple pass/fail assertion helper."""
    if condition:
        print(f"✅ PASS — {pass_msg}")
    else:
        print(f"❌ FAIL — {fail_msg}")
    return condition

# ── Feature engineering pipeline (no Flask dependency) ────────────────────────
def haversine_approx(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2) ** 2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def engineer_features(raw: dict) -> pd.DataFrame:
    r = raw.copy()
    dt = pd.to_datetime(r["trans_date_trans_time"])
    r["hour"]        = dt.hour
    r["day_of_week"] = dt.dayofweek
    r["month"]       = dt.month
    dob = pd.to_datetime(r["dob"])
    r["age"] = (dt - dob).days // 365
    r["distance_km"] = haversine_approx(r["lat"], r["long"], r["merch_lat"], r["merch_long"])
    r["log_amt"] = np.log1p(r["amt"])
    for col in ["trans_date_trans_time", "dob", "merchant", "city", "trans_num", "job"]:
        r.pop(col, None)
    df = pd.DataFrame([r])
    df = pd.get_dummies(df, columns=["category", "state"], drop_first=False, dtype=int)
    df = df.reindex(columns=feature_cols, fill_value=0)
    df[num_features] = scaler.transform(df[num_features])
    return df

# ── Two test transactions: obviously legit vs obviously fraudulent ─────────────
LEGIT_TX = {
    "trans_date_trans_time": "2024-06-15 13:00:00",  # midday Saturday
    "merchant": "Local Grocery",
    "category": "grocery_pos",
    "amt": 42.50,                                    # small, normal grocery run
    "city": "Denver",
    "state": "CO",
    "lat": 39.7392,
    "long": -104.9903,
    "city_pop": 500000,
    "job": "Teacher",
    "dob": "1985-06-15",
    "trans_num": "legit001",
    "merch_lat": 39.7500,                            # merchant 1.3 km away
    "merch_long": -104.9800,
}

FRAUD_TX = {
    "trans_date_trans_time": "2024-06-15 02:47:00",  # 2:47 AM
    "merchant": "Far Away Store",
    "category": "shopping_net",
    "amt": 4800.00,                                  # very high amount
    "city": "Denver",
    "state": "CO",
    "lat": 39.7392,
    "long": -104.9903,
    "city_pop": 500000,
    "job": "Teacher",
    "dob": "1985-06-15",
    "trans_num": "fraud001",
    "merch_lat": 48.8566,                            # Paris — 8,900 km away
    "merch_long": 2.3522,
}

print("✅ Setup complete — model, scaler, and feature pipeline loaded")
print(f"   Model type  : {type(model).__name__}")
print(f"   Feature cols: {len(feature_cols)}")

✅ Setup complete — model, scaler, and feature pipeline loaded
   Model type  : LGBMClassifier
   Feature cols: 39


---
## Test 15 — Prediction output is a valid class label (0 or 1)

`model.predict()` must return either `0` (legitimate) or `1` (fraud) for a well-formed input. Any other value — `None`, a float, an exception — means the pipeline is broken.

In [2]:
# Test 15 — predict() returns 0 or 1 for a valid input
df_legit = engineer_features(LEGIT_TX.copy())
prediction = model.predict(df_legit)[0]

check(prediction in (0, 1),
      f"predict() returned {prediction} — valid class label (0=legit, 1=fraud)",
      f"predict() returned {prediction!r} — expected 0 or 1")

check(isinstance(int(prediction), int),
      f"Prediction is an integer type",
      f"Prediction has unexpected type: {type(prediction)}")

print(f"\n   Legit transaction verdict: {'🚨 Fraud' if prediction == 1 else '✅ Legitimate'}")

✅ PASS — predict() returned 0 — valid class label (0=legit, 1=fraud)
✅ PASS — Prediction is an integer type

   Legit transaction verdict: ✅ Legitimate


---
## Test 16 — `predict_proba` output is in `[0.0, 1.0]`

`predict_proba()` returns a probability — it must always be between 0 and 1. A value outside this range would indicate a numerical problem in the model. We test with both the legit and fraud inputs to cover both ends of the probability range.

In [3]:
# Test 16 — predict_proba() output is in [0.0, 1.0] for both inputs
df_fraud = engineer_features(FRAUD_TX.copy())

proba_legit = model.predict_proba(df_legit)[0][1]   # fraud probability for legit tx
proba_fraud = model.predict_proba(df_fraud)[0][1]   # fraud probability for fraud tx

check(0.0 <= proba_legit <= 1.0,
      f"Legit tx fraud probability = {proba_legit:.4f} — within [0, 1]",
      f"Legit tx fraud probability = {proba_legit:.4f} — OUTSIDE [0, 1]")

check(0.0 <= proba_fraud <= 1.0,
      f"Fraud tx fraud probability = {proba_fraud:.4f} — within [0, 1]",
      f"Fraud tx fraud probability = {proba_fraud:.4f} — OUTSIDE [0, 1]")

print(f"\n   Legit tx  → fraud proba: {proba_legit:.4f}")
print(f"   Fraud tx  → fraud proba: {proba_fraud:.4f}")

✅ PASS — Legit tx fraud probability = 0.0000 — within [0, 1]
✅ PASS — Fraud tx fraud probability = 0.7165 — within [0, 1]

   Legit tx  → fraud proba: 0.0000
   Fraud tx  → fraud proba: 0.7165


---
## Test 17 — Predictions are deterministic

Given the same input, the model must always return the same output. ML models should be deterministic at inference time (no randomness). If the same transaction returns different fraud probabilities on repeated calls, that is a serious reliability bug — the model cannot be trusted in production.

In [4]:
# Test 17 — Determinism: same input → same prediction every time
results = []
for i in range(5):
    df_i = engineer_features(LEGIT_TX.copy())
    results.append(model.predict_proba(df_i)[0][1])

all_identical = len(set(round(r, 8) for r in results)) == 1

check(all_identical,
      f"All 5 repeated predictions identical: {results[0]:.8f}",
      f"Predictions varied across runs: {results} — model is non-deterministic")

print(f"\n   Repeated predictions: {[round(r, 6) for r in results]}")

✅ PASS — All 5 repeated predictions identical: 0.00002359

   Repeated predictions: [np.float64(2.4e-05), np.float64(2.4e-05), np.float64(2.4e-05), np.float64(2.4e-05), np.float64(2.4e-05)]


---
## Test 18 — Model is directionally sensible: fraud pattern scores higher than legit pattern

The fraud transaction was constructed with **three strong fraud signals**:
- Amount: **\$4,800** (very high — fraud-likely)
- Distance: **~8,900 km** (cardholder in Denver, merchant in Paris)
- Hour: **2:47 AM** (peak fraud hour per Section 2 EDA)

The legit transaction has a small amount (\$42.50), merchant 1.3 km away, and a midday timestamp.

The model **must** assign a higher fraud probability to the fraud-pattern input. If it does not, the model has not learned meaningful fraud signals and should not be deployed.

In [5]:
# Test 18 — Fraud-pattern input scores higher than legit-pattern input
print(f"   Legit transaction  → fraud probability : {proba_legit:.4f}")
print(f"   Fraud transaction  → fraud probability : {proba_fraud:.4f}")
print()

check(proba_fraud > proba_legit,
      f"Fraud pattern ({proba_fraud:.4f}) scored higher than legit pattern ({proba_legit:.4f}) — model is directionally sensible",
      f"Fraud pattern ({proba_fraud:.4f}) did NOT score higher than legit ({proba_legit:.4f}) — model may not have learned fraud signals correctly")

margin = proba_fraud - proba_legit
check(margin > 0.05,
      f"Margin between fraud and legit = {margin:.4f} (> 0.05 threshold — meaningful separation)",
      f"Margin between fraud and legit = {margin:.4f} — model barely distinguishes the two patterns")

   Legit transaction  → fraud probability : 0.0000
   Fraud transaction  → fraud probability : 0.7165

✅ PASS — Fraud pattern (0.7165) scored higher than legit pattern (0.0000) — model is directionally sensible
✅ PASS — Margin between fraud and legit = 0.7165 (> 0.05 threshold — meaningful separation)


np.True_

---
## Test 19 — Batch prediction: model handles multiple rows without error

In production the model will score many transactions at once. This test passes a batch of 5 diverse transactions (mix of legit and fraud patterns) and verifies:
- No crash on batch input
- Output shape is `(5,)` — one prediction per row
- All probabilities are in `[0, 1]`

This catches bugs where the model was accidentally saved as a single-row-only predictor.

In [6]:
# Test 19 — Batch of 5 transactions processed without error
batch_transactions = [
    {**LEGIT_TX,  "trans_num": "b1", "amt": 25.00,   "trans_date_trans_time": "2024-01-10 09:15:00"},
    {**FRAUD_TX,  "trans_num": "b2", "amt": 3200.00,  "trans_date_trans_time": "2024-01-10 03:22:00"},
    {**LEGIT_TX,  "trans_num": "b3", "amt": 15.99,   "trans_date_trans_time": "2024-01-10 12:00:00"},
    {**FRAUD_TX,  "trans_num": "b4", "amt": 990.00,   "trans_date_trans_time": "2024-01-10 01:05:00"},
    {**LEGIT_TX,  "trans_num": "b5", "amt": 67.40,   "trans_date_trans_time": "2024-01-10 17:45:00"},
]

try:
    batch_df = pd.concat([engineer_features(tx) for tx in batch_transactions], ignore_index=True)
    batch_probas = model.predict_proba(batch_df)[:, 1]
    batch_preds  = model.predict(batch_df)
    no_error = True
except Exception as e:
    no_error = False
    print(f"   Exception: {e}")

check(no_error,
      "Batch of 5 transactions processed without error",
      "Batch processing raised an exception")

if no_error:
    check(len(batch_probas) == 5,
          f"Output has 5 probabilities — one per input row",
          f"Output has {len(batch_probas)} probabilities — expected 5")

    all_valid = all(0.0 <= p <= 1.0 for p in batch_probas)
    check(all_valid,
          "All 5 batch probabilities are in [0, 1]",
          f"Some batch probabilities are out of range: {batch_probas}")

    print("\n   Batch results:")
    labels = ["Legit", "Fraud", "Legit", "Fraud", "Legit"]
    for i, (label, prob, pred) in enumerate(zip(labels, batch_probas, batch_preds)):
        verdict = '🚨 Fraud' if pred == 1 else '✅ Legit '
        print(f"   Tx {i+1} ({label:5s}) → fraud proba: {prob:.4f}  verdict: {verdict}")

✅ PASS — Batch of 5 transactions processed without error
✅ PASS — Output has 5 probabilities — one per input row
✅ PASS — All 5 batch probabilities are in [0, 1]

   Batch results:
   Tx 1 (Legit) → fraud proba: 0.0000  verdict: ✅ Legit 
   Tx 2 (Fraud) → fraud proba: 0.9764  verdict: 🚨 Fraud
   Tx 3 (Legit) → fraud proba: 0.0000  verdict: ✅ Legit 
   Tx 4 (Fraud) → fraud proba: 0.9764  verdict: 🚨 Fraud
   Tx 5 (Legit) → fraud proba: 0.0001  verdict: ✅ Legit 


---
## Phase 3 Summary

If all cells above printed `✅ PASS`, the model inference layer is working correctly:

| Test | What was verified |
|------|------------------|
| 15 | `predict()` returns a valid class label (0 or 1) |
| 16 | `predict_proba()` is always in `[0.0, 1.0]` |
| 17 | Same input always produces the same prediction (deterministic) |
| 18 | Fraud-pattern transaction scores higher than legit-pattern (directionally sensible) |
| 19 | Batch of 5 mixed transactions processed correctly with right output shape |

---

## Full Testing Suite Summary

| Phase | Notebook | Tests | Validates |
|-------|----------|-------|-----------|
| 1 | `Phase_1_testing.ipynb` | 1–5 | All artefact files load and have correct structure |
| 2 | `Phase_2_testing.ipynb` | 6–14 | Feature engineering math is correct on known inputs |
| 3 | `Phase_3_testing.ipynb` | 15–19 | Model inference is valid, deterministic, and directionally correct |

All 19 tests passing means the model pipeline is ready for submission and demo.